### Importação de bibliotecas 

In [1]:

from IPython import get_ipython

import pandas as pd 
import os
import dotenv
from pathlib import Path
from pandas import DataFrame

dotenv.load_dotenv()



try:
    from src.data.utils import gerar_df_dic,carregar_parquet_local
except ModuleNotFoundError:
    # Se falhar, instala o pacote local e importa
    get_ipython().run_line_magic('pip', 'install -e ..')
    from src.data.utils import gerar_df_dic,carregar_parquet_local

### Leitura de arquivos (CSV)

In [4]:
ano = 2023

#### TS_ALUNO <br>
> Informações dos alunos que participaram das avaliações estaduais do 2º ano do ensino fundamental.

In [ ]:
#raw_alunos, dicionario_alunos = gerar_df_dic(ano,'TS_ALUNO')
# raw_alunos.head(5)
# dicionario_alunos


Lendo dicionário em: ..\data\raw\dados_2023\DICIONÁRIO\Dicionario_Microdados_AEEB_2023.xlsx


#### TS_ITEM <br>
> Informações dos itens de provas utilizados na avaliação estadual do 2º ano do ensino fundamental para a disciplina Língua Portuguesa (LP).

In [ ]:
#raw_itens, dicionario_itens = gerar_df_dic(ano,'TS_ITEM')
# raw_itens.head(5)
# dicionario_itens.head(5)

Lendo dicionário em: ..\data\raw\dados_2023\DICIONÁRIO\Dicionario_Microdados_AEEB_2023.xlsx


#### TS_ESTADO

> Resultados das avaliações estaduais por UF do 2º ano do ensino fundamental para a disciplina Língua Portuguesa (LP), equalizadas com o SAEB.

In [ ]:
#raw_uf, dicionario_uf = gerar_df_dic(ano,'TS_ESTADO')
# raw_uf.head(5)
# dicionario_uf.head(5)


Lendo dicionário em: ..\data\raw\dados_2023\DICIONÁRIO\Dicionario_Microdados_AEEB_2023.xlsx


#### TS_MUNICIPIO 

> Resultados das avaliações estaduais por município do 2º ano do ensino fundamental para a disciplina Língua Portuguesa (LP), equalizadas com o SAEB.

In [ ]:
#raw_municipio, dicionario_municipio = gerar_df_dic(2023,'TS_MUNICIPIO')
# raw_municipio.head(5)
# dicionario_municipio.head(5)

Lendo dicionário em: ..\data\raw\dados_2023\DICIONÁRIO\Dicionario_Microdados_AEEB_2023.xlsx


### Analise da documentação do Inep (Leiame)

**raw_uf** <br>
*TS_ESTADO: Apresenta os resultados das avaliações estaduais do 2º ano do ensino fundamental para Língua Portuguesa (LP), também equalizados com o Saeb, mas neste arquivo os dados estão agregados por Estado (Unidade da Federação)*

**raw_municipio** <br>
*TS_MUNICIPIO : Contém os resultados das avaliações estaduais do 2º ano do ensino fundamental para a disciplina de Língua Portuguesa (LP) agregados em nível municipal
. Estes resultados são equalizados com os do Sistema de Avaliação da Educação Básica (Saeb)*

**raw_itens** <br>
*TS_ITEM : Contém o cadastro e as informações dos itens (questões) das provas utilizados na avaliação estadual do 2º ano para a disciplina de Língua Portuguesa*

**raw_alunos** <br> 
*TS_ALUNO:   é uma tabela de microdados que contém as informações dos alunos que participaram das avaliações estaduais do 2º ano do ensino fundamental os dados dos alunos seguem as leis de LGPD (Não há dados pessoais).*





**Breve descrição do que foi encontrado nos documentos técnicos LEIAME da base de dados**

Houve uma diferença no tratamento de qualidade de exclusão e inconsistência ao longo dos anos 
faremos uma breve analise de como isso foi tratado antes de nossa analise pelo Saeb: 

**2023**

- **TS_ALUNO** ->não existem dados de São Paulo na tabela <br><br>
- **TS_ALUNO** -> 4 registros foram retirados em razão de teremo 743 na proeficiencia (VL_PROFICIENCIA_LP) e 0 na maracação de alfabetização  <br><br>
- **TS_ITEM** -> A base foi gerada sem as informações de Mato Grosso do Sul, São Paulo e Sergipe 

**2024** 

- **TS_ALUNO** ->  Houve a exclusão de 224 registros de alunos de sete escolas (no Amazonas, Espírito Santo e São Paulo) que não foram localizadas no Censo Escolar 2024
. Foram excluídos também 67 registros devido à mesma inconsistência do ano anterior (proficiência 743, mas 0 em alfabetizado)
. Iniciado neste ano foi a dos códigos reais das escolas por máscaras, para seguir o padrão de proteção LGPD <br><br>
- **TS_MUNICIPIO** -> Foram retirados 3 registros onde a variável de percentual de aluno alfabetizado (PC_ALUNO_ALFABETIZADO) era igual a zero  <br><br>
- **TS_ITEM** ->  Itens da Bahia, São Paulo e Rio Grande do Sul foram retirados porque seus parâmetros não pareciam estar na escala do Saeb
. A Bahia também não enviou os itens comuns<br><br>
- **TS_ESTADO** -> Roraima (que não realizou a avaliação em 2024) e o Distrito Federal não enviaram dados
. Os resultados da rede estadual do Distrito Federal ficaram associados apenas à base TS_MUNICIPIO

**2025**

- **TS_ALUNO** ->Foram retirados 628 registros que estavam com a informação da escola em branco (Missing), ligados a dez escolas (em Rondônia, Pernambuco, Espírito Santo, São Paulo e Mato Grosso) que não foram achadas no Censo Escolar 2025
. A política de mascarar os códigos das escolas continuou a ser aplicada<br><br>

- **TS_MUNICIPIO** ->Foram retirados 7 registros com a variável PC_ALUNO_ALFABETIZADO zerada. <br><br>

- **TS_ITEM** -> Assim como no ano anterior, itens de São Paulo e do Rio Grande do Sul foram retirados por não estarem na escala correta do Saeb


### EDA

Para simplificar iremos carregar todos os dados a partir daqui já na camada bronze (.parquet) 
podemos ignorar a primeira leitura de arquivos a partir daqui caso já tenha sido rodado o pipeline_s3, irei fazer as analises todas locais num primeiro momento, depois irei integrar com a nuvem

In [41]:
ano = 2024

#dados
raw_alunos = carregar_parquet_local(ano,'TS_ALUNO')
raw_itens =  carregar_parquet_local(ano,'TS_ITEM') 
raw_uf  = carregar_parquet_local(ano,'TS_ESTADO') 
raw_municipio  = carregar_parquet_local(ano,'TS_MUNICIPIO') 


#Dicionarios
dicionario_alunos =   carregar_parquet_local(ano,'TS_ALUNO', ler_dicionario=True)
dicionario_itens =   carregar_parquet_local(ano,'TS_ITEM', ler_dicionario=True)
dicionario_municipio =   carregar_parquet_local(ano,'TS_ESTADO', ler_dicionario=True)
dicionario_uf =   carregar_parquet_local(ano,'TS_MUNICIPIO', ler_dicionario=True)

Lendo Parquet em: ..\data\bronze\ano=2024\dados\TS_ALUNO.parquet
Lendo Parquet em: ..\data\bronze\ano=2024\dados\TS_ITEM.parquet
Lendo Parquet em: ..\data\bronze\ano=2024\dados\TS_ESTADO.parquet
Lendo Parquet em: ..\data\bronze\ano=2024\dados\TS_MUNICIPIO.parquet
Lendo Parquet em: ..\data\bronze\ano=2024\dicionario\dicionario_TS_ALUNO.parquet
Lendo Parquet em: ..\data\bronze\ano=2024\dicionario\dicionario_TS_ITEM.parquet
Lendo Parquet em: ..\data\bronze\ano=2024\dicionario\dicionario_TS_ESTADO.parquet
Lendo Parquet em: ..\data\bronze\ano=2024\dicionario\dicionario_TS_MUNICIPIO.parquet


In [42]:
#Nao ocultar as informações de descrição 
pd.set_option('display.max_colwidth', None)

In [43]:
raw_municipio.isna().sum().sum()

0

In [44]:
raw_uf.isna().sum().sum()

0

In [45]:
raw_itens.isna().sum()

itens_na = pd.DataFrame(raw_itens.isna().sum(), columns=['Qtd_Nulos'])
itens_na['Descrição'] = dicionario_itens  .loc[itens_na.index, 'Descrição']

itens_na_filtrado = itens_na[itens_na['Qtd_Nulos'] > 0]
itens_na_filtrado


,Qtd_Nulos,Descrição
DS_GABARITO,459,Gabarito do Item
NU_PARAM_A,4,Parâmetro de discriminação: é o poder de discriminação do item para diferenciar os participantes que dominam dos participantes que não dominam a habilidade avaliada.
NU_PARAM_B,463,"Parâmetro de dificuldade: associado à dificuldade do item, sendo que quanto maior seu valor, mais difícil é o item."
NU_PARAM_C,463,Parâmetro de acerto ao acaso: é a probabilidade de um participante acertar o item não dominando a habilidade exigida.
NU_PARAM_B1,1212,Parâmetro de dificuldade da transição entre as categorias de “erro” e “acerto parcial”.
NU_PARAM_B2,1212,Parâmetro de dificuldade da transição entre as categorias de “acerto parcial” e “acerto total”.
NU_PARAM_B3,1416,Parâmetro de dificuldade da transição entre as categorias de “acerto parcial” e “acerto total”.
NU_PARAM_B4,1569,Parâmetro de dificuldade da transição entre as categorias de “acerto parcial” e “acerto total”.


In [ ]:
dicionario_alunos

,Nome da Variável,Descrição,Variáveis Categóricas,Tamanho,Tipo
1,NU_ANO_AVALIACAO,Ano de aplicação da avaliação estadual,nan,4,Numérica
2,CO_UF,Código da Unidade da Federação,nan,2,Numérica
3,SG_UF,Sigla da Unidade da Federação,nan,2,Alfanumérica
4,ID_ALUNO,Identificador do Aluno,nan,8,Numérica
5,TP_SERIE,Ano Escolar,2 - 2º ano do Ensino Fundamental,1,Numérica
6,ID_ESCOLA,Máscara do Códigoo da Escola (códigos fictícios),nan,8,Numérica
7,TP_DEPENDENCIA,Dependência Administrativa da Escola,1 - Federal;\n2 - Estadual;\n3 - Municipal;\n4 - Privada.,1,Numérica
8,CO_MUNICIPIO,Código do Município onde está localizada a escola.,nan,7,Numérica
9,NO_MUNICIPIO,Nome do Município onde está localizada a escola.,nan,150,Alfanumérica
10,IN_PRESENCA_LP,Indicador de presença na prova de Língua Portuguesa (LP). Cada UF seguirá a regra prevista para considerar um aluno presente na avaliação estadual.,0 - Ausente,1,Numérica


In [46]:
alunos_na = pd.DataFrame(raw_alunos.isna().sum(), columns=['Qtd_Nulos'])
alunos_na['Descrição'] = dicionario_alunos.loc[alunos_na.index, 'Descrição']

alunos_na_filtrado = alunos_na[alunos_na['Qtd_Nulos'] > 0]
alunos_na_filtrado


KeyError: "None of [Index(['NU_ANO_AVALIACAO', 'CO_UF', 'SG_UF', 'ID_ALUNO', 'TP_SERIE',\n       'ID_ESCOLA', 'TP_DEPENDENCIA', 'CO_MUNICIPIO', 'NO_MUNICIPIO',\n       'IN_PRESENCA_LP', 'IN_PREENCHIMENTO_LP', 'CO_CADERNO_LP',\n       'VL_PESO_ALUNO_LP', 'VL_PROFICIENCIA_LP', 'IN_ALFABETIZADO'],\n      dtype='object')] are in the [index]"

'Peso do aluno na prova de Língua Portuguesa'

In [12]:
raw_alunos

,NU_ANO_AVALIACAO,CO_UF,SG_UF,ID_ALUNO,TP_SERIE,ID_ESCOLA,TP_DEPENDENCIA,CO_MUNICIPIO,NO_MUNICIPIO,IN_PRESENCA_LP,IN_PREENCHIMENTO_LP,CO_CADERNO_LP,VL_PESO_ALUNO_LP,VL_PROFICIENCIA_LP,IN_ALFABETIZADO
0,2023,11,RO,11008701,2,60000001,3,1100205,Porto Velho,0,0,17,NaN,NaN,0
1,2023,11,RO,11008695,2,60000001,3,1100205,Porto Velho,1,1,17,1.045465,714.314857,0
2,2023,11,RO,11008687,2,60000001,3,1100205,Porto Velho,0,0,17,NaN,NaN,0
3,2023,11,RO,11008682,2,60000001,3,1100205,Porto Velho,1,1,17,1.045465,759.206313,1
4,2023,11,RO,11008729,2,60000001,3,1100205,Porto Velho,0,0,19,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1747434,2023,52,GO,52076808,2,60042811,3,5200258,Águas Lindas de Goiás,1,1,6,1.341165,721.771196,0
1747435,2023,52,GO,52076800,2,60042811,3,5200258,Águas Lindas de Goiás,1,1,6,1.341165,762.068013,1
1747436,2023,52,GO,52076799,2,60042811,3,5200258,Águas Lindas de Goiás,0,0,6,NaN,NaN,0
1747437,2023,52,GO,52076782,2,60042811,3,5200258,Águas Lindas de Goiás,0,0,6,NaN,NaN,0


In [11]:
alunos_nulos = raw_alunos[raw_alunos.isna().any(axis=1)]
alunos_nulos

,NU_ANO_AVALIACAO,CO_UF,SG_UF,ID_ALUNO,TP_SERIE,ID_ESCOLA,TP_DEPENDENCIA,CO_MUNICIPIO,NO_MUNICIPIO,IN_PRESENCA_LP,IN_PREENCHIMENTO_LP,CO_CADERNO_LP,VL_PESO_ALUNO_LP,VL_PROFICIENCIA_LP,IN_ALFABETIZADO
0,2023,11,RO,11008701,2,60000001,3,1100205,Porto Velho,0,0,17,NaN,NaN,0
2,2023,11,RO,11008687,2,60000001,3,1100205,Porto Velho,0,0,17,NaN,NaN,0
4,2023,11,RO,11008729,2,60000001,3,1100205,Porto Velho,0,0,19,NaN,NaN,0
7,2023,11,RO,11008726,2,60000001,3,1100205,Porto Velho,0,0,19,NaN,NaN,0
22,2023,11,RO,11008706,2,60000001,3,1100205,Porto Velho,0,0,18,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1747414,2023,52,GO,52076961,2,60042811,3,5200258,Águas Lindas de Goiás,0,0,11,NaN,NaN,0
1747415,2023,52,GO,52076947,2,60042811,3,5200258,Águas Lindas de Goiás,0,0,11,NaN,NaN,0
1747423,2023,52,GO,52076868,2,60042811,3,5200258,Águas Lindas de Goiás,0,0,8,NaN,NaN,0
1747436,2023,52,GO,52076799,2,60042811,3,5200258,Águas Lindas de Goiás,0,0,6,NaN,NaN,0
